In [3]:
import requests
from bs4 import BeautifulSoup
import csv  # 加上CSV库

url = "https://www.sony.com.cn/content/sonyportal/zh-cn/cms/newscenter/corporate/2023.html"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/37.36"
}

r = requests.get(url, headers=headers)
r.encoding = "utf-8"
soup = BeautifulSoup(r.text, "html.parser")

news_items = soup.select("ul.news_list1 li .news")

with open("sony_news_2023.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    # 写入表头
    writer.writerow(["日期", "新闻标题"])
    
    for i, item in enumerate(news_items, 1):
        title = item.get_text(strip=True)
        date = title[:10]       # 日期
        news_title = title[10:] # 标题
        
        # 打印到控制台
        print(f"序号：{i}")
        print(f"日期：{date}")
        print(f"标题：{news_title}")
        print("-" * 60)
        
        #  写入 CSV
        writer.writerow([date, news_title])

序号：1
日期：2023.12.29
标题：【微分享】索尼中国 第一届 大学生创想大赛 “企”梦索尼 招募正式启动！
------------------------------------------------------------
序号：2
日期：2023.11.17
标题：索尼宣布悠如音乐（YURU MUSIC）乐器创客马拉松中国赛正式启动
------------------------------------------------------------
序号：3
日期：2023.11.17
标题：2024年索尼影视娱乐公司庆祝哥伦比亚电影公司百年庆典 揭晓百年纪念标志
------------------------------------------------------------
序号：4
日期：2023.11.09
标题：索尼集团发布2023财年第二季度财报
------------------------------------------------------------
序号：5
日期：2023.11.07
标题：索尼于进博会首发新型环境技术“协生农法”智能协生APP
------------------------------------------------------------
序号：6
日期：2023.11.06
标题：植根中国，共筑可持续未来 索尼中国在进博会发布第十八份可持续发展报告
------------------------------------------------------------
序号：7
日期：2023.11.06
标题：用创意和科技的力量让音乐无处不在 索尼在进博会宣布将YURU MUSIC（悠如音乐）项目引进中国
------------------------------------------------------------
序号：8
日期：2023.11.06
标题：创新可持续，感动向未来 索尼亮相第六届中国国际进口博览会
------------------------------------------------------------
序号：9
日期：2023.10.26
标题：激发创新灵感与热情，与追梦者共启未来 第八届索尼中国创新大赛展会在沪举办
--------

In [2]:
import requests
from bs4 import BeautifulSoup
import csv
from urllib.parse import urljoin

# 目标 URL
BASE_URL = "https://www.sony.com/en/SonyInfo/News/Press/archive2023.html"

# 伪装成浏览器，防止被反爬虫拦截
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def scrape_sony_news():
    print(f"正在抓取: {BASE_URL} ...")
    
    try:
        # 1. 发送请求
        response = requests.get(BASE_URL, headers=HEADERS)
        response.raise_for_status() # 检查请求是否成功
        response.encoding = 'utf-8' # 设置编码

        # 2. 解析 HTML
        soup = BeautifulSoup(response.text, 'html.parser')

        # 3. 准备存储数据的列表
        news_data = []

        # 4. 找到所有的新闻区块 (按月份分组)
        # 注意：根据页面结构，我们查找月份标题和对应的列表
        months = soup.find_all('h2', class_='com_newsrelease-month')
        
        # 实际上页面结构是 h2(月份) -> ul(class_='com_newsrelease-area') -> li
        # 所以我们可以直接找所有的 li 项
        news_items = soup.find_all('li', class_='com_newsrelease-item')

        for item in news_items:
            # 提取日期
            date_tag = item.find('span', class_='com_newsrelease-date')
            date = date_tag.get_text(strip=True) if date_tag else "N/A"

            # 提取标题和链接
            link_tag = item.find('span', class_='com_newsrelease-link').find('a')
            title = link_tag.get_text(strip=True) if link_tag else "N/A"
            
            # 处理相对链接 -> 绝对链接
            relative_link = link_tag['href'] if link_tag else "N/A"
            full_link = urljoin(BASE_URL, relative_link)

            # 提取详情/摘要 (如果有的话)
            detail_tag = item.find('span', class_='com_newsrelease-detail')
            detail = detail_tag.get_text(strip=True) if detail_tag else "N/A"

            news_data.append({
                "date": date,
                "title": title,
                "link": full_link,
                "detail": detail
            })

        return news_data

    except requests.exceptions.RequestException as e:
        print(f"请求出错: {e}")
        return None

def save_to_csv(data, filename="sony_news_2023.csv"):
    if not data:
        print("没有数据可保存。")
        return

    # 写入 CSV 文件
    keys = data[0].keys()
    with open(filename, 'w', newline='', encoding='utf-8-sig') as f: # utf-8-sig 防止Excel中文乱码
        dict_writer = csv.DictWriter(f, keys)
        dict_writer.writeheader()
        dict_writer.writerows(data)
    print(f"数据已成功保存至 {filename}")

if __name__ == "__main__":
    news = scrape_sony_news()
    if news:
        print(f"成功抓取 {len(news)} 条新闻。")
        save_to_csv(news)

正在抓取: https://www.sony.com/en/SonyInfo/News/Press/archive2023.html ...
请求出错: 403 Client Error: Forbidden for url: https://www.sony.com/en/SonyInfo/News/Press/archive2023.html
